# Assignment 2 — Data preparation + portfolio structure check

This section prepares the two datasets used in the assignment:

- Fama–French factors: `rmrf`, `smb`, `hml`, `rf`
- 25 size–B/M portfolios (5×5)

What I do here:
1) load both files robustly (FF “CSVs” have headers/notes)
2) keep only the *monthly* value-weighted block for the 25 portfolios
3) convert returns from percent to decimal (if needed)
4) align everything on a common monthly index
5) compute portfolio excess returns \( r^i_t = R^i_t - rf_t \)

Before moving to regressions, I also verify the 5×5 ordering **directly from the file header**.

In [9]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from io import StringIO

## 1) Locate files

I auto-detect filenames so this works even if extensions are `.csv` vs `.CSV`.

In [10]:
factors_path = next(iter(Path("raw_data").glob("F-F_Research_Data_Factors.*")), None)
ports_path = next(iter(Path("raw_data").glob("25_Portfolios_5x5.*")), None)

if factors_path is None or ports_path is None:
    raise FileNotFoundError("Missing one of the required files: factors or 25 portfolios.")

print("Using these:")
print(" - factors:", factors_path.name)
print(" - portfolios:", ports_path.name)

Using these:
 - factors: F-F_Research_Data_Factors.CSV
 - portfolios: 25_Portfolios_5x5.CSV


## 2) Robust readers (FF-style)

FF files often contain text above the numeric table.
So I find the first row that starts with `YYYYMM,` and read from there.

For the portfolios file, I keep only the **first continuous monthly block** and stop
as soon as the pattern breaks (this avoids duplicated / stacked sections).

In [12]:
def find_first_yyyymm_line(path, pattern=r"^\s*\d{6}\s*,"):
    rx = re.compile(pattern)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            if rx.search(line):
                return i
    return None

def read_ff_factors(path):
    start = find_first_yyyymm_line(path)
    if start is None:
        raise ValueError("Couldn't find a YYYYMM data line in the factors file.")

    df = pd.read_csv(path, skiprows=start, header=None, engine="python").iloc[:, :5]
    df.columns = ["yyyymm", "rmrf", "smb", "hml", "rf"]

    df["yyyymm"] = df["yyyymm"].astype(str).str.strip()
    df = df[df["yyyymm"].str.fullmatch(r"\d{6}", na=False)].copy()

    for c in ["rmrf", "smb", "hml", "rf"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    return df

def read_first_monthly_block(path):
    start = find_first_yyyymm_line(path)
    if start is None:
        raise ValueError("Couldn't find a YYYYMM data line in the portfolios file.")

    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for _ in range(start):
            next(f)

        for line in f:
            line = line.strip()
            if not line:
                continue

            first = line.split(",")[0].strip()
            if re.fullmatch(r"\d{6}", first):
                rows.append(line)
            else:
                break

    return pd.read_csv(StringIO("\n".join(rows)), header=None, engine="python")

## 3) Portfolio structure verification (100% safe)

I check the header line right above the first `YYYYMM` row.
In this file, it explicitly says:

- “Average Value Weighted Returns — Monthly”
- columns are labelled `SMALL ... BIG` and `LoBM ... HiBM`

So the 5×5 ordering is directly given by the source file (no guessing).

In [13]:
start = find_first_yyyymm_line(ports_path)
with open(ports_path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()

# The line right above the first YYYYMM line contains the column labels
header_line = lines[start - 1].strip() if start and start > 0 else ""

def looks_like_header(line):
    return ("," in line) and (re.search(r"[A-Za-z]", line) is not None) and (len(line.split(",")) >= 10)

has_header = looks_like_header(header_line)

print("Header found:", has_header)
if has_header:
    header = [x.strip() for x in header_line.split(",")]
    # first field is empty in this file -> it corresponds to the date column
    header[0] = "yyyymm"
    print("Example labels:", header[1], "|", header[5], "|", header[-1])
else:
    header = None
    print("No usable header detected (would fall back to generic P01..P25 naming).")

Header found: True
Example labels: SMALL LoBM | SMALL HiBM | BIG HiBM


## 4) Load data using verified labels (preferred)

If the header is present, I use the real portfolio names (e.g., `SMALL LoBM`, `BIG HiBM`).
That makes the later 5×5 tables completely unambiguous.

In [14]:
# Load factors
factors_raw = read_ff_factors(factors_path)

# Load portfolios monthly block
ports_block = read_first_monthly_block(ports_path).iloc[:, :26]

if header is None or len(header) != ports_block.shape[1]:
    # fallback (shouldn't happen in your case, but safe)
    ports_block.columns = ["yyyymm"] + [f"P{i:02d}" for i in range(1, 26)]
else:
    ports_block.columns = header

# Basic peek
print("Factors loaded:", factors_raw.shape)
print("Portfolios monthly block loaded:", ports_block.shape)

display(factors_raw.head())
display(ports_block.head())

Factors loaded: (1170, 5)
Portfolios monthly block loaded: (1170, 26)


,yyyymm,rmrf,smb,hml,rf
0,192607,2.96,-2.56,-2.43,0.22
1,192608,2.64,-1.17,3.82,0.25
2,192609,0.36,-1.40,0.13,0.23
3,192610,-3.24,-0.09,0.70,0.32
4,192611,2.53,-0.10,-0.51,0.31


,yyyymm,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,...,ME4 BM1,ME4 BM2,ME4 BM3,ME4 BM4,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM
0,192607,5.8248,-1.7006,0.4875,-1.4580,2.0534,1.2077,2.4192,0.4926,-2.6049,...,1.5893,1.5278,1.2978,0.2727,2.4678,3.4539,6.0902,2.0266,3.1111,0.5623
1,192608,-2.0206,-8.0282,1.3796,1.4606,8.3968,2.3618,-1.1849,4.0084,0.5038,...,1.3336,3.8730,2.0021,2.1706,5.3422,1.0124,4.1903,2.0131,5.4849,7.7576
2,192609,-4.8291,-2.6154,-4.3417,-3.2729,0.8649,-2.6540,-1.2618,1.0829,-3.5480,...,1.0923,-0.5250,-1.7636,1.4646,0.8730,-1.2906,3.6538,0.0950,-0.7487,-2.4284
3,192610,-9.3729,-3.5519,-3.4948,3.4413,-2.5476,-2.8069,-3.2663,-5.0745,-8.0191,...,-3.3361,-2.6559,-2.1070,-3.1051,-5.3525,-2.7413,-3.0071,-2.2437,-4.6719,-5.8129
4,192611,5.5888,4.1877,2.4623,-4.4494,0.5362,3.1033,-2.3690,3.0078,5.1546,...,3.4448,2.3887,3.7335,4.9320,1.8213,4.2946,2.5326,1.5204,3.6619,2.5636


## 5) Build monthly index + convert units

Dates are stored as `YYYYMM`. I convert to a proper monthly datetime index.

FF returns are usually in percent, so I convert to decimals when magnitudes suggest it.

In [21]:
def yyyymm_to_month_start(s):
    s = s.astype(str).str.strip()
    dt = pd.to_datetime(s + "01", format="%Y%m%d", errors="coerce")
    return dt.dt.to_period("M").dt.to_timestamp()

def to_decimal_if_percent(df, threshold=0.5):
    num = df.select_dtypes(include=[np.number])
    med = num.stack().abs().median()
    out = df.copy()
    converted = False
    if pd.notna(med) and med > threshold:
        out[num.columns] = out[num.columns] / 100.0
        converted = True
    return out, converted, float(med)

# Factors
factors = factors_raw.copy()
factors["date"] = yyyymm_to_month_start(factors["yyyymm"])
factors = factors.set_index("date").drop(columns=["yyyymm"]).sort_index()

# Portfolios
ports = ports_block.copy()
ports["date"] = yyyymm_to_month_start(ports["yyyymm"])
ports = ports.set_index("date").drop(columns=["yyyymm"]).sort_index()

# Duplicate date check (should be zero)
print("Duplicate dates:")
print(" - factors:", factors.index.duplicated().sum())
print(" - portfolios:", ports.index.duplicated().sum())

# Convert % -> decimal if needed
factors, conv_f, med_f = to_decimal_if_percent(factors)
ports, conv_p, med_p = to_decimal_if_percent(ports)

print(f"\nConverted factors to decimals: {conv_f} (median |x| = {med_f:.4f})")
print(f"Converted portfolios to decimals: {conv_p} (median |x| = {med_p:.4f})")

print("\nFactor columns:", factors.columns.tolist())
print("Portfolio columns (first 5):", ports.columns.tolist()[:5])
print("Portfolio columns (last 3):", ports.columns.tolist()[-3:])

Duplicate dates:
 - factors: 0
 - portfolios: 0

Converted factors to decimals: True (median |x| = 1.2300)
Converted portfolios to decimals: True (median |x| = 3.6699)

Factor columns: ['rmrf', 'smb', 'hml', 'rf']
Portfolio columns (first 5): ['SMALL LoBM', 'ME1 BM2', 'ME1 BM3', 'ME1 BM4', 'SMALL HiBM']
Portfolio columns (last 3): ['ME5 BM3', 'ME5 BM4', 'BIG HiBM']


## 6) Align samples + compute excess portfolio returns

We align on the intersection of dates and compute:

\[
r^i_t = R^i_t - rf_t
\]

At the end of this cell, the two main objects are:
- `factors_a`  (rmrf, smb, hml, rf)
- `ports_excess` (25 portfolio excess returns)

In [17]:
common_idx = factors.index.intersection(ports.index)
factors_a = factors.loc[common_idx].copy()
ports_a = ports.loc[common_idx].copy()

rf = factors_a["rf"]
ports_excess = ports_a.sub(rf, axis=0)

print("Common sample:", common_idx.min(), "→", common_idx.max(), "| T =", len(common_idx))

print("\nSanity checks:")
print("Average rf:", float(rf.mean()))
print("Average rmrf:", float(factors_a["rmrf"].mean()))
print("Average excess portfolio return:", float(ports_excess.mean().mean()))

# Quick check on extremes (pick any 3 portfolios by name)
sample_cols = ports_excess.columns[:3].tolist()
display(ports_excess[sample_cols].describe().T)

Common sample: 1926-07-01 00:00:00 → 2023-12-01 00:00:00 | T = 1170

Sanity checks:
Average rf: 0.002676666666666667
Average rmrf: 0.006780683760683762
Average excess portfolio return: 0.00888564177777778


,count,mean,std,min,25%,50%,75%,max
SMALL LoBM,1170.0,0.005606,0.119998,-0.495030,-0.049951,0.004699,0.053457,1.478101
ME1 BM2,1170.0,0.006792,0.097099,-0.352972,-0.036855,0.004698,0.049126,1.260068
ME1 BM3,1170.0,0.010045,0.091883,-0.339091,-0.030663,0.009921,0.045843,1.020371


## Checkpoint

Data is now clean and ready for Question (i).

- factors: `factors_a`
- portfolio excess returns: `ports_excess`
- portfolio names come from the FF header (value-weighted, monthly, 5×5 size–B/M labels)

In [29]:
print("Ready for regressions (finally)!!")
print("T =", len(factors_a))
print("Factors:", factors_a.columns.tolist())
print("Number of portfolios:", ports_excess.shape[1])

Ready for regressions (finally)!!
T = 1170
Factors: ['rmrf', 'smb', 'hml', 'rf']
Number of portfolios: 25


# Question (i) - CAPM time-series regressions on the 25 portfolios (1964–1993)

Now that the data is clean, we run the standard CAPM time-series regression for each portfolio:

\[
r^i_t = \alpha_i + \beta_i \cdot rmrf_t + \varepsilon^i_t
\]

- \(r^i_t\): excess return of portfolio \(i\) (already computed as \(R^i_t - rf_t\))
- \(rmrf_t\): market excess return from the FF factors file

I focus on the sample **Jan 1964 -> Jan 1993** (same window as in the original FF setup).
For each portfolio, I store:
- mean monthly excess return
- alpha and its t-stat
- beta and its t-stat
- adjusted \(R^2\)

Finally, I display everything in 5×5 tables using the **actual FF column labels**
(SMALL -> BIG by rows, LoBM -> HiBM by columns), so there’s no ambiguity about ordering.

In [9]:
import statsmodels.api as sm
import pandas as pd
import numpy as np

## 1) Slice the 1964–1993 sample (monthly)

I take the intersection of dates just to be extra safe.

In [10]:
def slice_monthly(df, start, end):
    start = pd.to_datetime(start).to_period("M").to_timestamp()
    end = pd.to_datetime(end).to_period("M").to_timestamp()
    return df.loc[(df.index >= start) & (df.index <= end)].copy()

start, end = "1964-01-01", "1993-01-01"

factors_6493 = slice_monthly(factors_a, start, end)
ports_6493 = slice_monthly(ports_excess, start, end)

common_idx = factors_6493.index.intersection(ports_6493.index)
factors_6493 = factors_6493.loc[common_idx]
ports_6493 = ports_6493.loc[common_idx]

print("Sample range:", common_idx.min(), "→", common_idx.max())
print("T =", len(common_idx))
print("Aligned:", factors_6493.index.equals(ports_6493.index))

Sample range: 1964-01-01 00:00:00 → 1993-01-01 00:00:00
T = 349
Aligned: True


## 2) Run 25 CAPM regressions and store results

One regression per portfolio, same regressor everywhere:
- constant
- `rmrf`

(Everything is in monthly decimals.)

In [11]:
def run_capm_time_series(Y: pd.DataFrame, rmrf: pd.Series) -> pd.DataFrame:
    X = sm.add_constant(rmrf.rename("rmrf"))
    rows = []

    for name in Y.columns:
        y = Y[name]
        fit = sm.OLS(y, X, missing="drop").fit()

        rows.append({
            "portfolio": name,
            "mean_excess": float(y.mean()),
            "alpha": float(fit.params["const"]),
            "alpha_t": float(fit.tvalues["const"]),
            "beta": float(fit.params["rmrf"]),
            "beta_t": float(fit.tvalues["rmrf"]),
            "adj_r2": float(fit.rsquared_adj),
        })

    return pd.DataFrame(rows).set_index("portfolio")

capm_6493 = run_capm_time_series(ports_6493, factors_6493["rmrf"])
capm_6493.head()

,mean_excess,alpha,alpha_t,beta,beta_t,adj_r2
portfolio,,,,,,
SMALL LoBM,0.003090,-0.002637,-1.126317,1.425884,27.798851,0.689224
ME1 BM2,0.007425,0.002403,1.186556,1.250530,28.182754,0.695076
ME1 BM3,0.007840,0.003186,1.726040,1.158853,28.657677,0.702122
ME1 BM4,0.009437,0.005112,2.793956,1.077028,26.867342,0.674418
SMALL HiBM,0.010844,0.006414,3.081514,1.103020,24.185137,0.626578


## 3) Build 5×5 tables using the FF labels (no guessing)

The portfolio names look like:
- `SMALL LoBM`, `SMALL HiBM`
- `ME2 BM1` ... `ME4 BM5`
- `BIG LoBM`, `BIG HiBM`

So I use the label patterns to place each portfolio into the correct cell:
- rows: `SMALL`, `ME2`, `ME3`, `ME4`, `BIG`
- cols: `LoBM`/`BM1`, `BM2`, `BM3`, `BM4`, `HiBM`/`BM5`

This way, even if the raw columns were re-ordered, the 5×5 grid stays correct.

In [12]:
size_order = ["SMALL", "ME2", "ME3", "ME4", "BIG"]
size_names = ["Small", "2", "3", "4", "Big"]

bm_order = ["LoBM", "BM2", "BM3", "BM4", "HiBM"]
bm_names = ["Low", "2", "3", "4", "High"]

def parse_ff_portfolio_label(label: str):
    """
    Returns (size_bucket, bm_bucket) from labels like:
    - 'SMALL LoBM' -> ('SMALL', 'LoBM')
    - 'ME3 BM4' -> ('ME3', 'BM4')
    - 'BIG HiBM' -> ('BIG', 'HiBM')

    Note: ME rows use BM1/BM5 sometimes, which correspond to LoBM/HiBM.
    """
    parts = label.split()
    if len(parts) != 2:
        return None

    size, bm = parts[0], parts[1]

    if size not in {"SMALL", "ME1", "ME2", "ME3", "ME4", "ME5", "BIG"}:
        return None

    # Normalize size naming: treat ME1 as SMALL row, ME5 as BIG row (FF conventions)
    if size == "ME1":
        size = "SMALL"
    if size == "ME5":
        size = "BIG"

    # Normalize BM endpoints
    if bm == "BM1":
        bm = "LoBM"
    if bm == "BM5":
        bm = "HiBM"

    return size, bm

def to_5x5(res_df: pd.DataFrame, col: str) -> pd.DataFrame:
    tbl = pd.DataFrame(np.nan, index=size_names, columns=bm_names)

    for port in res_df.index:
        parsed = parse_ff_portfolio_label(port)
        if parsed is None:
            continue
        size, bm = parsed

        if (size in size_order) and (bm in bm_order):
            i = size_order.index(size)
            j = bm_order.index(bm)
            tbl.iloc[i, j] = res_df.loc[port, col]

    return tbl

tables_6493 = {
    "Mean excess return": to_5x5(capm_6493, "mean_excess"),
    "Beta (rmrf)": to_5x5(capm_6493, "beta"),
    "t(Beta)": to_5x5(capm_6493, "beta_t"),
    "Alpha": to_5x5(capm_6493, "alpha"),
    "t(Alpha)": to_5x5(capm_6493, "alpha_t"),
    "Adj R^2": to_5x5(capm_6493, "adj_r2"),
}

tables_6493["Beta (rmrf)"]

,Low,2,3,4,High
Small,1.425884,1.250530,1.158853,1.077028,1.103020
2,1.428427,1.234521,1.116015,1.033107,1.123314
3,1.353789,1.162876,1.033577,0.975694,1.069104
4,1.223638,1.128797,1.034855,0.973928,1.081046
Big,1.001999,0.982546,0.866932,0.834828,0.870462


## 4) Display the CAPM results (1964–1993)

I show the full set of 5×5 tables in decimals.

Then, for readability, I also show mean excess returns and alphas in monthly percent.

In [13]:
for name, tbl in tables_6493.items():
    print("\n" + name)
    display(tbl)


Mean excess return


,Low,2,3,4,High
Small,0.003090,0.007425,0.007840,0.009437,0.010844
2,0.004047,0.006582,0.008828,0.009565,0.010755
3,0.004491,0.006975,0.007049,0.008792,0.010202
4,0.004657,0.004035,0.006177,0.008066,0.009194
Big,0.003263,0.003467,0.003490,0.005139,0.006125



Beta (rmrf)


,Low,2,3,4,High
Small,1.425884,1.250530,1.158853,1.077028,1.103020
2,1.428427,1.234521,1.116015,1.033107,1.123314
3,1.353789,1.162876,1.033577,0.975694,1.069104
4,1.223638,1.128797,1.034855,0.973928,1.081046
Big,1.001999,0.982546,0.866932,0.834828,0.870462



t(Beta)


...

,Low,2,3,4,High
Small,-0.002637,0.002403,0.003186,0.005112,0.006414
2,-0.001689,0.001625,0.004346,0.005416,0.006243
3,-0.000945,0.002305,0.002898,0.004873,0.005908
4,-0.000257,-0.000498,0.002021,0.004155,0.004853
Big,-0.000761,-0.000479,0.000008,0.001786,0.002629



t(Alpha)


,Low,2,3,4,High
Small,-1.126317,1.186556,1.726040,2.793956,3.081514
2,-0.962518,1.079549,3.037569,3.991288,3.663913
3,-0.697838,1.981224,2.549935,4.272519,3.722447
4,-0.254490,-0.552534,2.130326,3.693725,3.305077
Big,-0.826358,-0.625438,0.008073,1.718931,1.760606



Adj R^2


,Low,2,3,4,High
Small,0.689224,0.695076,0.702122,0.674418,0.626578
2,0.798485,0.801022,0.784406,0.776091,0.722094
3,0.856597,0.856670,0.831826,0.814020,0.730655
4,0.897678,0.903559,0.876877,0.817672,0.764250
Big,0.876410,0.907838,0.821534,0.794213,0.670104


### Optional: percent view (presentation only)

This is the same information, just multiplied by 100 for:
- mean excess returns
- alphas

Betas, t-stats, and \(R^2\) stay unchanged.

In [14]:
print("Mean excess return (%)")
display(tables_6493["Mean excess return"] * 100)

print("\nAlpha (%)")
display(tables_6493["Alpha"] * 100)

Mean excess return (%)


,Low,2,3,4,High
Small,0.308968,0.742515,0.783957,0.943715,1.084405
2,0.404740,0.658241,0.882838,0.956548,1.075462
3,0.449140,0.697523,0.704935,0.879180,1.020204
4,0.465701,0.403485,0.617670,0.806612,0.919441
Big,0.326350,0.346699,0.348962,0.513917,0.612476



Alpha (%)


,Low,2,3,4,High
Small,-0.263673,0.240296,0.318556,0.511176,0.641427
2,-0.168923,0.162452,0.434641,0.541647,0.624334
3,-0.094548,0.230506,0.289846,0.487337,0.590847
4,-0.025718,-0.049845,0.202068,0.415478,0.485288
Big,-0.076058,-0.047897,0.000798,0.178646,0.262894


## Quick takeaway (so I don’t get lost)

Once the tables are printed, the typical patterns to look for are:
- mean excess returns increasing from Low -> High B/M (value effect)
- betas generally decreasing from Small -> Big (often, but not always)
- whether alphas are systematically different from zero across the grid

I’ll use these tables to answer the written part of Question (i).

# Question (i) 

Here I test the single-factor CAPM using monthly data for the 25 Fama–French size–book-to-market portfolios.

For each portfolio \(i\), I run the time-series regression:

\[
r^i_t = \alpha_i + \beta_i \cdot rmrf_t + \varepsilon^i_t
\]

where:
- \(r^i_t\) is the portfolio **excess return** (already computed as \(R^i_t - rf_t\)),
- \(rmrf_t\) is the market excess return from the factors file.

I use the standard FF window **Jan 1964 -> Jan 1993**, then I report:
- mean monthly excess return
- \(\alpha\), \(\beta\), their t-stats
- adjusted \(R^2\)

Finally, I reshape the results into 5×5 tables using the **actual portfolio labels**
(SMALL→BIG by rows, Low→High B/M by columns), so ordering is unambiguous.

In [15]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

## 1) Subsample: 1964–1993 (monthly)

I slice both the factors and the portfolio excess returns on the same dates
and double-check that the indices match perfectly.

In [16]:
def slice_monthly(df, start, end):
    start = pd.to_datetime(start).to_period("M").to_timestamp()
    end = pd.to_datetime(end).to_period("M").to_timestamp()
    return df.loc[(df.index >= start) & (df.index <= end)].copy()

start, end = "1964-01-01", "1993-01-01"

factors_6493 = slice_monthly(factors_a, start, end)
ports_6493 = slice_monthly(ports_excess, start, end)

# Align on the intersection, just to be safe
common_idx = factors_6493.index.intersection(ports_6493.index)
factors_6493 = factors_6493.loc[common_idx]
ports_6493 = ports_6493.loc[common_idx]

print("Sample range:", common_idx.min(), "→", common_idx.max())
print("T =", len(common_idx))
print("Aligned:", factors_6493.index.equals(ports_6493.index))

Sample range: 1964-01-01 00:00:00 → 1993-01-01 00:00:00
T = 349
Aligned: True


## 2) Run the 25 CAPM time-series regressions

One regression per portfolio:
- regress \(r^i_t\) on a constant and \(rmrf_t\)
- store mean excess return, alpha/beta, t-stats, and adj \(R^2\)

In [17]:
def run_capm(Y: pd.DataFrame, rmrf: pd.Series) -> pd.DataFrame:
    X = sm.add_constant(rmrf.rename("rmrf"))
    rows = []

    for name in Y.columns:
        y = Y[name]
        fit = sm.OLS(y, X, missing="drop").fit()

        rows.append({
            "portfolio": name,
            "mean_excess": float(y.mean()),
            "alpha": float(fit.params["const"]),
            "alpha_t": float(fit.tvalues["const"]),
            "beta": float(fit.params["rmrf"]),
            "beta_t": float(fit.tvalues["rmrf"]),
            "adj_r2": float(fit.rsquared_adj),
        })

    return pd.DataFrame(rows).set_index("portfolio")

capm_6493 = run_capm(ports_6493, factors_6493["rmrf"])
display(capm_6493.head())

,mean_excess,alpha,alpha_t,beta,beta_t,adj_r2
portfolio,,,,,,
SMALL LoBM,0.003090,-0.002637,-1.126317,1.425884,27.798851,0.689224
ME1 BM2,0.007425,0.002403,1.186556,1.250530,28.182754,0.695076
ME1 BM3,0.007840,0.003186,1.726040,1.158853,28.657677,0.702122
ME1 BM4,0.009437,0.005112,2.793956,1.077028,26.867342,0.674418
SMALL HiBM,0.010844,0.006414,3.081514,1.103020,24.185137,0.626578


## 3) Reshape results into 5×5 tables (using the labels)

The portfolio names encode the grid:
- Size buckets: `SMALL`, `ME2`, `ME3`, `ME4`, `BIG`  (rows)
- B/M buckets: `LoBM`, `BM2`, `BM3`, `BM4`, `HiBM`  (columns)

Note: the FF file uses `ME1` for SMALL and `ME5` for BIG in some columns,
and it uses `BM1`/`BM5` as endpoints. I normalize those to LoBM/HiBM.

In [18]:
size_order = ["SMALL", "ME2", "ME3", "ME4", "BIG"]
size_names = ["Small", "2", "3", "4", "Big"]

bm_order = ["LoBM", "BM2", "BM3", "BM4", "HiBM"]
bm_names = ["Low", "2", "3", "4", "High"]

def parse_ff_label(label: str):
    parts = label.split()
    if len(parts) != 2:
        return None

    size, bm = parts[0], parts[1]

    # normalize size endpoints
    if size == "ME1":
        size = "SMALL"
    if size == "ME5":
        size = "BIG"

    # normalize B/M endpoints
    if bm == "BM1":
        bm = "LoBM"
    if bm == "BM5":
        bm = "HiBM"

    return size, bm

def to_5x5(res_df: pd.DataFrame, col: str) -> pd.DataFrame:
    tbl = pd.DataFrame(np.nan, index=size_names, columns=bm_names)

    for port in res_df.index:
        parsed = parse_ff_label(port)
        if parsed is None:
            continue

        size, bm = parsed
        if (size in size_order) and (bm in bm_order):
            i = size_order.index(size)
            j = bm_order.index(bm)
            tbl.iloc[i, j] = res_df.loc[port, col]

    return tbl

tables_6493 = {
    "Mean excess return": to_5x5(capm_6493, "mean_excess"),
    "Beta (rmrf)": to_5x5(capm_6493, "beta"),
    "t(Beta)": to_5x5(capm_6493, "beta_t"),
    "Alpha": to_5x5(capm_6493, "alpha"),
    "t(Alpha)": to_5x5(capm_6493, "alpha_t"),
    "Adj R^2": to_5x5(capm_6493, "adj_r2"),
}

display(tables_6493["Mean excess return"])

,Low,2,3,4,High
Small,0.003090,0.007425,0.007840,0.009437,0.010844
2,0.004047,0.006582,0.008828,0.009565,0.010755
3,0.004491,0.006975,0.007049,0.008792,0.010202
4,0.004657,0.004035,0.006177,0.008066,0.009194
Big,0.003263,0.003467,0.003490,0.005139,0.006125


## 4) Display all CAPM tables

Everything below is in decimals (monthly).
Then I also show mean excess returns and alphas in % per month for readability.

In [19]:
for name, tbl in tables_6493.items():
    print("\n" + name)
    display(tbl)


Mean excess return


,Low,2,3,4,High
Small,0.003090,0.007425,0.007840,0.009437,0.010844
2,0.004047,0.006582,0.008828,0.009565,0.010755
3,0.004491,0.006975,0.007049,0.008792,0.010202
4,0.004657,0.004035,0.006177,0.008066,0.009194
Big,0.003263,0.003467,0.003490,0.005139,0.006125



Beta (rmrf)


,Low,2,3,4,High
Small,1.425884,1.250530,1.158853,1.077028,1.103020
2,1.428427,1.234521,1.116015,1.033107,1.123314
3,1.353789,1.162876,1.033577,0.975694,1.069104
4,1.223638,1.128797,1.034855,0.973928,1.081046
Big,1.001999,0.982546,0.866932,0.834828,0.870462



t(Beta)


...

,Low,2,3,4,High
Small,-0.002637,0.002403,0.003186,0.005112,0.006414
2,-0.001689,0.001625,0.004346,0.005416,0.006243
3,-0.000945,0.002305,0.002898,0.004873,0.005908
4,-0.000257,-0.000498,0.002021,0.004155,0.004853
Big,-0.000761,-0.000479,0.000008,0.001786,0.002629



t(Alpha)


,Low,2,3,4,High
Small,-1.126317,1.186556,1.726040,2.793956,3.081514
2,-0.962518,1.079549,3.037569,3.991288,3.663913
3,-0.697838,1.981224,2.549935,4.272519,3.722447
4,-0.254490,-0.552534,2.130326,3.693725,3.305077
Big,-0.826358,-0.625438,0.008073,1.718931,1.760606



Adj R^2


,Low,2,3,4,High
Small,0.689224,0.695076,0.702122,0.674418,0.626578
2,0.798485,0.801022,0.784406,0.776091,0.722094
3,0.856597,0.856670,0.831826,0.814020,0.730655
4,0.897678,0.903559,0.876877,0.817672,0.764250
Big,0.876410,0.907838,0.821534,0.794213,0.670104


### Presentation version: monthly percent

This is just formatting:
- mean excess returns and alphas ×100
- betas, t-stats, and \(R^2\) unchanged

In [20]:
print("Mean excess return (%)")
display(tables_6493["Mean excess return"] * 100)

print("\nAlpha (%)")
display(tables_6493["Alpha"] * 100)

Mean excess return (%)


,Low,2,3,4,High
Small,0.308968,0.742515,0.783957,0.943715,1.084405
2,0.404740,0.658241,0.882838,0.956548,1.075462
3,0.449140,0.697523,0.704935,0.879180,1.020204
4,0.465701,0.403485,0.617670,0.806612,0.919441
Big,0.326350,0.346699,0.348962,0.513917,0.612476



Alpha (%)


,Low,2,3,4,High
Small,-0.263673,0.240296,0.318556,0.511176,0.641427
2,-0.168923,0.162452,0.434641,0.541647,0.624334
3,-0.094548,0.230506,0.289846,0.487337,0.590847
4,-0.025718,-0.049845,0.202068,0.415478,0.485288
Big,-0.076058,-0.047897,0.000798,0.178646,0.262894


Table (i) reports the results of CAPM time-series regressions for the 25 Fama–French size–book-to-market portfolios over the period January 1964 to January 1993.

A clear pattern emerges in average excess returns. For a given size group, mean returns increase monotonically from low to high book-to-market portfolios. For example, among small stocks, the average monthly excess return rises from about 0.31% for low B/M portfolios to more than 1.08% for high B/M portfolios. A similar value pattern is present across all size categories, while the size effect itself is weaker and less systematic.

Estimated market betas are all positive and highly statistically significant, with t-statistics well above conventional thresholds. Betas tend to be higher for small portfolios and lower for large portfolios, indicating that smaller firms are more exposed to market risk. This pattern is broadly consistent with standard asset-pricing intuition.

Despite the strong explanatory power of the market factor, the CAPM alphas are not zero for many portfolios. Alphas are generally close to zero (and often slightly negative) for low book-to-market portfolios, but become positive and economically meaningful for high book-to-market portfolios. Several high B/M portfolios exhibit alphas that are statistically significant, suggesting systematic pricing errors under the CAPM.

Finally, adjusted R^2 values are relatively high, typically ranging from about 0.65 to above 0.90, indicating that the market factor explains a large share of the time-series variation in returns. However, the presence of significant alphas shows that a good fit in the time series does not imply correct pricing in the cross-section.

Overall, these results suggest that while the CAPM captures market exposure reasonably well, it fails to explain the cross-sectional variation in average returns, particularly the value premium observed across book-to-market portfolios.

# Question (ii)

Here we just read the 5×5 table of **mean monthly excess returns** and describe what it’s doing.

Quick map of the grid:
- **Rows (North -> South)**: Small -> Big (size increases as you go down)
- **Columns (East -> West)**: Low B/M -> High B/M (book-to-market increases as you go right)

So when we talk about “direction”, we literally mean moving around that table.

In [21]:
# Mean monthly excess returns (in % per month for readability)
mean_excess_pct = tables_6493["Mean excess return"] * 100
display(mean_excess_pct.round(3))

,Low,2,3,4,High
Small,0.309,0.743,0.784,0.944,1.084
2,0.405,0.658,0.883,0.957,1.075
3,0.449,0.698,0.705,0.879,1.020
4,0.466,0.403,0.618,0.807,0.919
Big,0.326,0.347,0.349,0.514,0.612


## What I see in the table

**Main pattern:** returns increase very clearly as you move **to the right** (Low -> High B/M).  
That’s the strong “value” gradient in this sample.

- Example (Small row): ~0.31% -> ~1.08% per month from Low to High B/M.
- Example (Big row): ~0.33% -> ~0.61% per month from Low to High B/M.

**Size direction (top → bottom):** the size pattern is weakest and not perfectly monotonic.
If anything, the really strong story here is value, not size.

So if I had to answer in one line:
> Average excess returns mostly rise **Westward** (towards high book-to-market).

In [30]:
# A tiny numeric summary to back up the "Westward" statement:
# compute (High - Low) spread in each size row (in % per month)
spread_west = (mean_excess_pct["High"] - mean_excess_pct["Low"]).rename("High-Low spread (%)")
display(spread_west.round(3))

print("Average High-Low spread across rows (% per month):", float(spread_west.mean()))

Small    0.775
2        0.671
3        0.571
4        0.454
Big      0.286
Name: High-Low spread (%), dtype: float64

Average High-Low spread across rows (% per month): 0.5514179369627507


### Interpretation of the patterns in average excess returns (1964–1993)

Looking at the 5×5 table of mean monthly **excess** returns, the strongest pattern is clearly along the **book-to-market** dimension. As you move from **Low B/M to High B/M** (left -> right), average excess returns increase almost everywhere in the grid. For example, for small stocks the mean excess return goes from roughly **0.31% per month** (Low) to about **1.08% per month** (High). Even for big stocks the same direction holds, just with a smaller spread (around **0.33% -> 0.61%** per month).

Along the **size** dimension (top -> bottom), things are much less clean: returns don’t move in a perfectly monotonic way when you hold B/M fixed. So if we had to summarise it simply, average excess returns mostly rise **westward**, towards high book-to-market portfolios. Across all size groups, High B/M portfolios earn higher average excess returns than Low B/M portfolios, so the biggest systematic increase is along **book-to-market**, not size.

This naturally raises the question of whether CAPM betas move enough to explain this return spread.

# Question (iii)

From (ii), average excess returns vary a lot across the 5×5 grid, especially moving from
**Low B/M to High B/M**.

The CAPM would say: portfolios with higher average excess returns should simply have
higher exposure to the market factor, i.e. higher **β**. So the natural check is:

- do the portfolios with higher mean returns also have higher CAPM betas?
- in particular, does the **value spread** (High B/M − Low B/M) show up as a **beta spread**?

If mean returns move a lot but betas don’t move much (or don’t move in the right direction),
then the spread in average returns is a CAPM “puzzle”.

In [23]:
# Re-show the two key tables side by side (mean returns in % for readability)
mean_pct = (tables_6493["Mean excess return"] * 100).round(3)
beta_tbl = tables_6493["Beta (rmrf)"].round(3)

display(mean_pct)
display(beta_tbl)

,Low,2,3,4,High
Small,0.309,0.743,0.784,0.944,1.084
2,0.405,0.658,0.883,0.957,1.075
3,0.449,0.698,0.705,0.879,1.020
4,0.466,0.403,0.618,0.807,0.919
Big,0.326,0.347,0.349,0.514,0.612


,Low,2,3,4,High
Small,1.426,1.251,1.159,1.077,1.103
2,1.428,1.235,1.116,1.033,1.123
3,1.354,1.163,1.034,0.976,1.069
4,1.224,1.129,1.035,0.974,1.081
Big,1.002,0.983,0.867,0.835,0.870


## A simple check: High–Low spreads within each size row

Here I compute, for each size group (row):
- High–Low mean return spread (in % per month)
- High–Low beta spread

I think this is the most direct way to answer the question without overcomplicating it.

In [24]:
hl_mean = (mean_pct["High"] - mean_pct["Low"]).rename("High-Low mean (%/month)")
hl_beta = (beta_tbl["High"] - beta_tbl["Low"]).rename("High-Low beta")

hl_check = pd.concat([hl_mean, hl_beta], axis=1)
display(hl_check)

print("Average High–Low mean spread (%/month):", float(hl_mean.mean()))
print("Average High–Low beta spread:", float(hl_beta.mean()))

,High-Low mean (%/month),High-Low beta
Small,0.775,-0.323
2,0.670,-0.305
3,0.571,-0.285
4,0.453,-0.143
Big,0.286,-0.132


Average High–Low mean spread (%/month): 0.5509999999999999
Average High–Low beta spread: -0.23760000000000003


### So is the spread in average returns a puzzle?

The spread in average excess returns across the 25 portfolios does represent a puzzle from the perspective of the CAPM.

From Question (ii), we observed a strong increase in average excess returns when moving from **Low to High book-to-market** portfolios. The High–Low return spread is economically large, averaging around **0.55% per month** across size groups.

However, this pattern is **not explained by market risk** as measured by the CAPM βs. Within each size category, High B/M portfolios tend to have **lower market betas** than Low B/M portfolios. The High–Low beta spread is negative for all size groups, with an average close to **−0.24**.

According to the CAPM, higher expected returns should be associated with higher market risk. Here, portfolios that earn higher average returns do not exhibit higher exposure to the market factor. In fact, the relationship goes in the opposite direction.

Therefore, the return spread itself is not the puzzle; the puzzle arises when we compare this spread to the pattern of CAPM βs. The CAPM fails to explain why High book-to-market portfolios earn higher average excess returns despite having lower market risk.

This failure of the CAPM to account for the systematic return differences across portfolios naturally raises the question of whether additional risk factors are needed to explain these patterns, which we investigate in the next question.

# Question (iv) 

This question is mostly about reading the CAPM regression outputs from (i).

What we want to check:
1) how many CAPM alphas are statistically significant (at 5% and 1%)
2) where they tend to show up in the 5×5 grid (e.g., value side vs growth side)
3) whether the adjusted R² values are generally high or low, and what that implies

We’ll keep the code minimal: just a couple of quick counts + a short summary table.

In [25]:
# We already have: capm_6493 (portfolio-level regression output)
# and tables_6493 (5×5 tables including t(Alpha) and Adj R^2)

alpha_t = capm_6493["alpha_t"].dropna()
n_5 = int((alpha_t.abs() >= 1.96).sum())
n_1 = int((alpha_t.abs() >= 2.58).sum())

print(f"Significant alphas at 5%  (|t|>=1.96): {n_5} / {len(alpha_t)}")
print(f"Significant alphas at 1%  (|t|>=2.58): {n_1} / {len(alpha_t)}")

# Show the most extreme alphas (by |t|)
top_alpha = capm_6493.loc[alpha_t.abs().sort_values(ascending=False).head(8).index,
                          ["alpha", "alpha_t", "beta", "adj_r2", "mean_excess"]].copy()

top_alpha["alpha_%"] = top_alpha["alpha"] * 100
top_alpha["mean_excess_%"] = top_alpha["mean_excess"] * 100

display(top_alpha[["mean_excess_%", "alpha_%", "alpha_t", "beta", "adj_r2"]].round(4))

Significant alphas at 5%  (|t|>=1.96): 12 / 25
Significant alphas at 1%  (|t|>=2.58): 9 / 25


,mean_excess_%,alpha_%,alpha_t,beta,adj_r2
portfolio,,,,,
ME3 BM4,0.8792,0.4873,4.2725,0.9757,0.8140
ME2 BM4,0.9565,0.5416,3.9913,1.0331,0.7761
ME3 BM5,1.0202,0.5908,3.7224,1.0691,0.7307
ME4 BM4,0.8066,0.4155,3.6937,0.9739,0.8177
ME2 BM5,1.0755,0.6243,3.6639,1.1233,0.7221
ME4 BM5,0.9194,0.4853,3.3051,1.0810,0.7643
SMALL HiBM,1.0844,0.6414,3.0815,1.1030,0.6266
ME2 BM3,0.8828,0.4346,3.0376,1.1160,0.7844


## Alpha significance: what it tells us

A statistically significant CAPM alpha means:  
even after controlling for market risk (β), the portfolio earns an average excess return
that is systematically too high (positive α) or too low (negative α) relative to what the CAPM predicts.

So significant α's are basically **pricing errors** for the CAPM.

In our results, significant alphas appear mainly on the **high book-to-market (value) side** of the table.
That matches what we saw earlier: average returns increase strongly from Low -> High B/M,
but betas do not increase in the same way. The CAPM therefore leaves a structured pattern in the residual
average returns, which shows up as non-zero α.

In [31]:
# Quick look at the 5×5 t(Alpha) table and Adj R^2 table
print("t(Alpha)")
display(tables_6493["t(Alpha)"].round(3))

print("\nAdjusted R^2")
display(tables_6493["Adj R^2"].round(3))

# Optional: summarize the distribution of adjusted R^2
r2 = capm_6493["adj_r2"].dropna()
print("\nAdj R^2 summary:")
display(r2.describe().round(3))

t(Alpha)


,Low,2,3,4,High
Small,-1.126,1.187,1.726,2.794,3.082
2,-0.963,1.080,3.038,3.991,3.664
3,-0.698,1.981,2.550,4.273,3.722
4,-0.254,-0.553,2.130,3.694,3.305
Big,-0.826,-0.625,0.008,1.719,1.761



Adjusted R^2


,Low,2,3,4,High
Small,0.689,0.695,0.702,0.674,0.627
2,0.798,0.801,0.784,0.776,0.722
3,0.857,0.857,0.832,0.814,0.731
4,0.898,0.904,0.877,0.818,0.764
Big,0.876,0.908,0.822,0.794,0.670



Adj R^2 summary:


count    25.000
mean      0.788
std       0.081
min       0.627
25%       0.722
50%       0.798
75%       0.857
max       0.908
Name: adj_r2, dtype: float64

### Significance of CAPM α and goodness of fit

We now examine whether the CAPM fully explains the excess returns of the 25 size–value portfolios by looking at the estimated αs and the adjusted R² from the time-series regressions.

#### Significance of α

A substantial fraction of portfolios exhibit statistically significant alphas:
- **12 out of 25** portfolios have |t(α)| ≥ 1.96 (significant at the 5% level),
- **9 out of 25** remain significant at the 1% level.

The significant alphas are mainly concentrated among **high book-to-market (value) portfolios**, especially in the middle and upper size groups. In contrast, low book-to-market portfolios tend to have alphas closer to zero and are generally not statistically significant.

This indicates that, even after controlling for market risk, the CAPM fails to fully account for the average returns of several portfolios. In other words, the model leaves **systematic pricing errors**, particularly along the value dimension.

#### Adjusted R²

The adjusted R² values are relatively high across portfolios, with:
- an average adjusted R² of about **0.79**,
- most values lying between **0.70 and 0.90**.

This shows that the market factor explains a large share of the **time-series variation** in returns. However, a high R² does not imply correct pricing: the presence of significant alphas means that the CAPM can fit returns well over time while still failing to explain their **average levels**.

Overall, these results suggest that the CAPM captures market comovement reasonably well, but is incomplete as an asset-pricing model for these portfolios.